# Face Anonymization from 5 Given Landmarks

Pure-geometric face removal for Einstar photogrammetry scans. Given the 5
anatomical landmarks (Nz, Iz, Cz, Lpa, Rpa), `anonymize_scan` strips the face
and reverts the result back to the raw Einstar (`crs=digitized`) frame so the
saved `.obj` + `.tsv` are a drop-in replacement for the original at
co-registration time.

Flow: load &rarr; pick 5 landmarks &rarr; `anonymize_scan` &rarr; save.


In [10]:
import logging
import os

import numpy as np
import pyvista as pv
import xarray as xr
from PIL import Image
import trimesh

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.geometry.photogrammetry.anonymization import (
    anonymize_scan,
    save_anonymized_scan,
)

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === Options ===
SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'
SCAN_PATH = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
OUT_PATH = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}_anon.obj'

# True: run the picker. False: use cached landmarks (fill in cell below).
INTERACTIVE = True

# anonymize_scan tunables (mm). Defaults match the library's recommended values.
ANON_OPTIONS = dict(
    head_isolation_radius_mm=220.0,
    cap_band_width_mm=15.0,
    cap_bin_size_mm=1.0,
    cap_foot_grad_threshold=0.2,
    cap_z_ceiling_mm=40.0,
    eyebrow_offset_mm=10.0,
    ear_delete_radius_mm=40.0,
    landmark_keep_radius_mm=8.0,
)


## Load the Einstar scan


In [11]:
surface = cedalion.io.read_einstar_obj(SCAN_PATH)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# trimesh 4.6 places the texture image on visual.material.image; if neither
# path has it, attach it from the sibling JPG so save_anonymized_scan can
# sanitize the final texture.
visual = surface.mesh.visual
img = getattr(visual, 'image', None) or getattr(getattr(visual, 'material', None), 'image', None)
if img is None:
    jpg = SCAN_PATH.replace('.obj', '.jpg')
    uv = getattr(visual, 'uv', None)
    assert os.path.exists(jpg) and uv is not None, 'no texture available'
    surface.mesh.visual = trimesh.visual.TextureVisuals(
        uv=uv, image=Image.open(jpg).convert('RGBA'),
    )


Loaded: 1,284,667 vertices, 2,223,716 faces


## Pick the 5 landmarks

Right-click on the mesh to place a sphere; click a sphere to cycle its label
through `Nz -> Iz -> Cz -> Lpa -> Rpa`. Close the window when all 5 are placed.
Skip this cell if `INTERACTIVE = False`.


In [12]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = cedalion.vis.blocks.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True,
    )
    pvplt.show()


Widget(value='<iframe src="http://localhost:42585/index.html?ui=P_0x7f2e70c57590_2&reconnect=auto" class="pyvi…

## Wrap picked points into a `LabeledPoints`


In [13]:
if INTERACTIVE:
    landmarks = get_landmarks()
else:
    # Cached fallback: paste coordinates from a previous interactive run here.
    landmark_labels = ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa']
    landmark_coordinates = np.asarray([
        [0.0, 0.0, 0.0],   # Nz  -- replace
        [0.0, 0.0, 0.0],   # Iz  -- replace
        [0.0, 0.0, 0.0],   # Cz  -- replace
        [0.0, 0.0, 0.0],   # Lpa -- replace
        [0.0, 0.0, 0.0],   # Rpa -- replace
    ])
    landmarks = xr.DataArray(
        np.vstack(landmark_coordinates),
        dims=['label', 'digitized'],
        coords={
            'label': ('label', landmark_labels),
            'type': ('label', [cdc.PointType.LANDMARK] * 5),
            'group': ('label', ['L'] * 5),
        },
    ).pint.quantify('mm')

labels = list(landmarks['label'].values)
assert set(labels) == {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}, f'bad labels: {labels}'
display(landmarks)


Magnitude,[[67.3260028810771 -11.451158171365705 478.3475193380646] [27.13790911247986 217.01868087023644 457.0993099960403] [181.98669298817038 116.53328682331225 429.7640407051109] [-7.758955014371736 60.47001712735505 415.085399568561] [75.93565646921411 77.90628633010931 563.5794111384439]]
Units,millimeter


## Anonymize


In [14]:
surface_anon, landmarks_anon = anonymize_scan(surface, landmarks, **ANON_OPTIONS)
print(f'S{SUBJECT_NUMBER}: {surface.nvertices:,} -> {surface_anon.nvertices:,} verts '
      f'(-{surface.nvertices - surface_anon.nvertices:,})')


S17: 1,284,667 -> 593,963 verts (-690,704)


## Compare original vs anonymized

Both meshes live in the `digitized` frame, so the side-by-side view shares one
coordinate system. The same five landmark spheres are overlaid on each.


In [15]:
lm_colors = {
    'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan',
    'Lpa': 'orange', 'Rpa': 'blue', 'LPA': 'orange', 'RPA': 'blue',
}
n_removed = surface.nvertices - surface_anon.nvertices

orig_vtk = trimesh_to_vtk_polydata(surface.mesh)
anon_vtk = trimesh_to_vtk_polydata(surface_anon.mesh)

lm_pos = landmarks_anon.pint.dequantify().values
lm_lbls = landmarks_anon['label'].values

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
pvplt.add_mesh(pv.wrap(orig_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(
    f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
    position='upper_left', font_size=14,
)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:42585/index.html?ui=P_0x7f2e3cfa5d50_3&reconnect=auto" class="pyvi…

## Save


In [16]:
written = save_anonymized_scan(surface_anon, OUT_PATH, landmarks=landmarks_anon)
for p in written:
    print(f'wrote {p}')


wrote /home/ma7/BA/PG_Subjects/Subject17/Subject17_anon.jpg
wrote /home/ma7/BA/PG_Subjects/Subject17/Subject17_anon.mtl
wrote /home/ma7/BA/PG_Subjects/Subject17/Subject17_anon.obj
wrote /home/ma7/BA/PG_Subjects/Subject17/Subject17_anon_landmarks.tsv
